In [1]:
!pip install -q transformers datasets scikit-learn torch evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report
from torch.utils.data import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
ds = load_dataset("gplsi/fake_job_postings_balanced_en")
df = ds["train"].to_pandas()
print(df.columns.tolist())
print(df.shape)
print(df["fraudulent"].value_counts())

# Combine text fields into one input string
text_cols = [c for c in ["title", "company_profile", "description", "requirements", "benefits"] if c in df.columns]
df["text"] = df[text_cols].fillna("").agg(" ".join, axis=1)
df["label"] = df["fraudulent"].astype(int)

df = df[["text", "label"]].dropna()
print(df.head())
print(df["label"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

balanced.csv:   0%|          | 0.00/4.34M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1732 [00:00<?, ? examples/s]

['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
(1732, 18)
fraudulent
0    866
1    866
Name: count, dtype: int64
                                                text  label
0  Initiativbewerbung Wir sind ein junges Berline...      0
1  Process Engineer Mechanical                   ...      1
2  Production Trading Floor Support  Triangle Wor...      0
3  Quality Engineer   Quality Assurance: 100% Emp...      1
4  Subsea Process System Engineer  Corporate over...      1
label
0    866
1    866
Name: count, dtype: int64


In [4]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df["label"]
)

In [5]:
vectorizer = TfidfVectorizer(
    max_features=8000, sublinear_tf=True, ngram_range=(1, 2), stop_words="english"
)
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])

svm = LinearSVC(class_weight="balanced", random_state=SEED)
svm_calibrated = CalibratedClassifierCV(svm)
svm_calibrated.fit(X_train, train_df["label"])

svm_preds = svm_calibrated.predict(X_test)

print("\n=== BASELINE: TF-IDF + LinearSVC ===")
print(classification_report(test_df["label"], svm_preds, target_names=["Genuine", "Fraudulent"], digits=4))



=== BASELINE: TF-IDF + LinearSVC ===
              precision    recall  f1-score   support

     Genuine     0.9050    0.9310    0.9178       174
  Fraudulent     0.9286    0.9017    0.9150       173

    accuracy                         0.9164       347
   macro avg     0.9168    0.9164    0.9164       347
weighted avg     0.9168    0.9164    0.9164       347



In [6]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(
    list(train_df["text"]), truncation=True, padding=True, max_length=256
)
test_encodings = tokenizer(
    list(test_df["text"]), truncation=True, padding=True, max_length=256
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
class JobScamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = JobScamDataset(train_encodings, list(train_df["label"]))
test_dataset = JobScamDataset(test_encodings, list(test_df["label"]))

In [8]:
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "Genuine", 1: "Fraudulent"},
    label2id={"Genuine": 0, "Fraudulent": 1},
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
training_args = TrainingArguments(
    output_dir="./job_scam_distilbert",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.326873,0.321829
2,0.185740,0.205818
3,0.135012,0.211041


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=261, training_loss=0.2574128196200078, metrics={'train_runtime': 40.1381, 'train_samples_per_second': 103.517, 'train_steps_per_second': 6.503, 'total_flos': 275201020707840.0, 'train_loss': 0.2574128196200078, 'epoch': 3.0})

In [10]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

print("\n=== DISTILBERT FINE-TUNED ===")
print(classification_report(true_labels, preds, target_names=["Genuine", "Fraudulent"], digits=4))

print("\n=== COMPARISON ===")
print("Run both classification_report outputs above side by side to compare macro-F1.")


=== DISTILBERT FINE-TUNED ===
              precision    recall  f1-score   support

     Genuine     0.9349    0.9080    0.9213       174
  Fraudulent     0.9101    0.9364    0.9231       173

    accuracy                         0.9222       347
   macro avg     0.9225    0.9222    0.9222       347
weighted avg     0.9225    0.9222    0.9222       347


=== COMPARISON ===
Run both classification_report outputs above side by side to compare macro-F1.


In [11]:
SAVE_DIR = "./distilbert_job_scam_final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"\nModel saved to {SAVE_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./distilbert_job_scam_final
